# 113 — CPTAC-3 External Cohort Construction

## Objective

Construct an **outcome-blind, case-level external-cohort eligibility ledger** from the audited CPTAC-3 acquisition and the frozen TCGA discovery reference. Reuse the approved G2 candidate policy (1,082 uniquely mapped primary RNA cases); adjudicate G3 donor evidence conservatively and record the G4–G8 technical readiness gates.

This notebook does **not** score frozen programs, infer resistance, evaluate structural replication, or freeze an external cohort. Unavailable or unreviewed source evidence remains unresolved.

## Inputs and reproducibility

- Notebook 111 local audit tables under `data/interim/metadata/`; no repeat download or raw-payload integrity sweep.
- Frozen registered TCGA `phase2.203.final_sample_mapping` mapping, verified against the registry SHA-256.
- Optional **reviewed** donor, lineage and QC evidence CSVs with explicit fields; no automatic web/GDC retrieval.
- Interim output ledger and concise readiness summary in `data/interim/metadata/`.

**Interpretive status:** G2 approved; G3 remains a prospective draft, not a validated donor-independence claim. This notebook requires a fresh **Run All** after formatting changes.

In [ ]:
# =============================================================================
# Imports, fixed paths, and local utilities
# =============================================================================

import json

import pandas as pd
from IPython.display import display

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths

ROOT = Paths.root
META = Paths.metadata
OUT = META
OUT.mkdir(parents=True, exist_ok=True)
REGISTRY = ROOT / "config" / "artifact_registry.json"
INDEX = META / "111_cptac3_file_index.csv"
PROVISIONAL = META / "111_cptac3_provisional_primary_tumor_files.csv"
G3_EVIDENCE = META / "113_reviewed_donor_evidence.csv"  # optional external, case-level reviewed evidence
G4_CROSSWALK = META / "113_reviewed_source_lineage_crosswalk.csv"  # optional reviewed source-stratum crosswalk
QC_EVIDENCE = META / "113_reviewed_qc_evidence.csv"  # optional per-file QC evaluation, source-method documented

def need(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required audited input missing: {path}")
    return path

def load(path):
    return pd.read_csv(need(path), dtype = "string").fillna(pd.NA)

def exact_unique(df, keys, label):
    if df.duplicated(keys).any():
        raise ValueError(f"{label}: duplicate keys {keys}")

def clean_id(x):
    return x.astype("string").str.strip().str.casefold()

def sha(path):
    return calculate_sha256(path)

In [ ]:
# =============================================================================
# Load audited CPTAC-3 inputs and verify frozen TCGA reference
# =============================================================================

with need(REGISTRY).open(encoding="utf-8") as handle:
    registry = json.load(handle)
tcga_ref = registry["artifacts"]["phase2.203.final_sample_mapping"]
assert tcga_ref["status"] == "frozen" and tcga_ref["shape"][0] == 9965
TCGA = need(ROOT / tcga_ref["path"])
assert sha(TCGA) == tcga_ref["sha256"], "Frozen TCGA mapping SHA-256 mismatch"
source = load(INDEX)
primary = load(PROVISIONAL)
tcga = load(TCGA)

print(
    "Audited inputs:", len(source), "file index rows;",
    len(primary), "provisional primary files;",
    len(tcga), "frozen TCGA cases",
)

## G2 — Approved primary RNA candidate universe

A single donor can contribute only one uniquely linked primary RNA file. The 288 multiplicity cases remain outside the primary universe. The 37 cases with *additional* ambiguous source files are **flags**, not automatic exclusions. All starting counts and distinct analysis units are checked before any downstream adjudication.

In [ ]:
# =============================================================================
# Reconstruct the approved G2 candidate ledger
# =============================================================================

required = {"modality", "file_id", "case_uuid", "case_submitter_id", "sample_uuid", "aliquot_uuid", "mapping_status", "sample_type", "primary_site", "disease_type", "project_id", "provider_md5"}
assert required.issubset(source.columns), required-set(source.columns)
assert required.issubset(primary.columns), required-set(primary.columns)
exact_unique(source, ["modality", "file_id"], "111 source")
exact_unique(primary, ["modality", "file_id"], "111 provisional primary")
assert len(source.loc[source.modality.eq("rnaseq")])==2968
assert len(source.loc[source.modality.eq("methylation")])==2610
rna = source.loc[source.modality.eq("rnaseq")].copy()
candidate = primary.loc[primary.modality.eq("rnaseq")].copy()
assert len(candidate)==1731 and candidate.case_uuid.nunique()==1370, "111 G2 source universe changed"
assert candidate.file_id.isin(rna.file_id).all()
assert candidate.mapping_status.eq("unique_aliquot_sample").all()
assert candidate.sample_type.eq("Primary Tumor").all()
assert candidate[["file_id", "case_uuid", "sample_uuid", "aliquot_uuid"]].notna().all().all()
case_count = candidate.groupby("case_uuid").agg(n_files = ("file_id", "nunique"), n_samples = ("sample_uuid", "nunique"), n_aliquots = ("aliquot_uuid", "nunique"))

simple_cases = case_count.loc[(case_count.n_files==1)&(case_count.n_samples==1)&(case_count.n_aliquots==1)].index
assert len(simple_cases)==1082 and len(case_count)==1370
assert len(case_count)-len(simple_cases)==288
ledger = candidate.loc[candidate.case_uuid.isin(simple_cases)].copy().sort_values("case_uuid").reset_index(drop = True)
assert len(ledger)==1082 and ledger.case_uuid.is_unique
ambig = rna.loc[rna.mapping_status.eq("ambiguous_sample_ancestry")]
assert len(ambig)==314
flagged = set(ambig.case_uuid.dropna()) & set(ledger.case_uuid)
assert len(flagged)==37, f"Additional ambiguous-file flags changed: {len(flagged)}"
ledger["other_ambiguous_rna_flag"] = ledger.case_uuid.isin(flagged)
ledger["g2_status"] = "SIMPLE_PRIMARY_CANDIDATE"
mult = case_count.loc[~case_count.index.isin(simple_cases)]
assert len(mult.loc[mult.n_samples>1])==276 and len(mult.loc[(mult.n_samples==1)&(mult.n_aliquots>1)])==12
assert mult.n_files.sum()==649
print("G2 exact:", len(ledger), "simple cases;", len(mult), "multiplicity cases;", len(flagged), "flagged; no biological data used")

## G3 — Proportionate cohort-level independence assessment

CPTAC-3 prospective Discovery/Confirmatory is treated as an external source cohort relative to frozen TCGA discovery, on the basis of documented prospective acquisition and published external-validation precedent. **This is a study-design assumption, not a patient-by-patient certification.** The fixed GDC `CPTAC-3` origin, PDC cohort labels where available, and completed exact-identifier collision screen are used to detect exceptions. The `CPTAC-TCGA` retrospective population is excluded; unexpected PDC projects, contradictory identifiers, and positive overlap evidence require review. A missing PDC match alone does not negate the directly audited GDC origin.

G3 is distinct from G4–G8 technical eligibility and does not itself freeze or authorize the analytic cohort. No expression results inform these decisions.

In [ ]:
# =============================================================================
# Screen comparable TCGA and CPTAC-3 identifier namespaces
# =============================================================================

## Identity namespace screening is descriptive; no automatic donor equivalence from UUID strings.
for c in ["rna_case_uuid", "case_submitter_id"]:
    if c not in tcga.columns:
        raise ValueError(f"TCGA frozen mapping lacks expected field {c}")

for c in ["case_uuid", "case_submitter_id"]:
    if ledger[c].isna().any():
        raise ValueError(f"CPTAC G2 missing {c}")
# A single common string can flag a collision only; donor linkage requires separate evidence.
tcga_uuids = set(clean_id(tcga["rna_case_uuid"]).dropna())
tcga_submitters = set(clean_id(tcga["case_submitter_id"]).dropna())

ledger["uuid_collision_flag"] = clean_id(ledger.case_uuid).isin(tcga_uuids)
ledger["submitter_collision_flag"] = clean_id(ledger.case_submitter_id).isin(tcga_submitters)
ledger["g3_class"] = "UNRESOLVED_INDEPENDENCE"
ledger["g3_evidence_id"] = pd.NA
ledger["g3_reason"] = "No reviewed, applicable donor-nonoverlap/overlap evidence"
ledger["g3_reviewed"] = False

print("Lexical collision flags (NOT donor-level evidence):", int(ledger.uuid_collision_flag.sum()), int(ledger.submitter_collision_flag.sum()))

evidence_sources = {}


## Original-cohort linkage — source contexts versus verified participants

The GDC `CPTAC-3` project identifies a **program-level source**, not necessarily the originating proteogenomic publication for every patient. GDC case `submitter_id` corresponds to the study participant identifier under the GDC/dbGaP submission model, but this does **not** by itself map a patient to a particular PDC study, original publication, or guarantee non-overlap with TCGA. See [GDC identifier model](https://docs.gdc.cancer.gov/Data/Data_Model/GDC_Data_Model/) and [NCI CPTAC accessions](https://gdc.cancer.gov/about-gdc/contributed-genomic-data-cancer-research/clinical-proteomic-tumor-analysis-consortium-cptac).

This section constructs two *different* outputs: (1) complete, unaltered case-to-source-context membership, plus candidate publication context **without asserting study membership**, and (2) exact participant-to-original-cohort links only if an external, reviewed crosswalk is available.

Optional `data/interim/metadata/113_reviewed_original_cohort_links.csv` schema:

`case_submitter_id,original_cohort,original_study_id,source_url,participant_id,identity_namespace,reviewed`

Use one row per source participant (one-to-many source studies are possible and remain separate). `reviewed=true` requires checking that the external participant identifier is really the **same identity namespace** as the CPTAC GDC submitter ID; the document must directly support study membership. Cases lacking such a link retain `UNLINKED_ORIGINAL_COHORT`. Study membership is **not** equivalent to G3 donor independence.

Potential publication contexts for investigation include LUAD, LUSC, ccRCC, UCEC, PDAC, HNSCC, and GBM, but source site/disease labels cannot individually prove these study assignments.

In [ ]:
# =============================================================================
# Define original-cohort linkage sources and candidate study context
# =============================================================================

COHORT_LINKS = META / "113_reviewed_original_cohort_links.csv"

# References are investigation leads, not a definitive crosswalk.
ORIGINAL_COHORT_SOURCES = {
    "LUAD": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7373300/",
    "LUSC": "https://pmc.ncbi.nlm.nih.gov/articles/PMC8475722/",
    "ccRCC": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7331093/",
    "UCEC": "https://pmc.ncbi.nlm.nih.gov/articles/PMC10631452/",
    "PDAC": "https://pmc.ncbi.nlm.nih.gov/articles/PMC8654574/",
    "HNSCC": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7946781/",
    "GBM": "https://pmc.ncbi.nlm.nih.gov/articles/PMC8044053/",
}

site_hints = {
    "Bronchus and lung": "LUAD or LUSC — verify histology",
    "Kidney": "ccRCC candidate — verify subtype",
    "Uterus, NOS": "UCEC candidate — verify diagnosis",
    "Pancreas": "PDAC candidate — verify diagnosis",
    "Brain": "GBM candidate — verify diagnosis",
}

ledger["original_cohort"] = pd.NA
ledger["original_study_id"] = pd.NA
ledger["original_cohort_source_url"] = pd.NA
ledger["original_cohort_link_status"] = "UNLINKED_ORIGINAL_COHORT"
ledger["original_cohort_hint"] = ledger["primary_site"].map(site_hints).fillna(
    "Study assignment requires external participant crosswalk"
)

# A descriptive source context is not a participant-to-publication link.
source_context_inventory = (
    ledger.groupby(["project_id", "primary_site", "disease_type"], dropna=False)
    .agg(
        n_g2_candidates=("case_uuid", "nunique"),
        n_primary_rna_files=("file_id", "nunique"),
    )
    .reset_index()
)

display(source_context_inventory)
print("Original-cohort links require verified external participant identifiers.")


In [ ]:
# =============================================================================
# Reconcile exact independently reviewed original-cohort participant links
# =============================================================================

original_cohort_link_table = pd.DataFrame(
    columns=[
        "case_uuid", "case_submitter_id", "original_cohort",
        "original_study_id", "source_url", "participant_id",
        "identity_namespace", "reviewed",
    ]
)

if COHORT_LINKS.exists():
    links = load(COHORT_LINKS)
    required_link = {
        "case_submitter_id", "original_cohort", "original_study_id",
        "source_url", "participant_id", "identity_namespace", "reviewed",
    }

    if not required_link.issubset(links.columns):
        raise ValueError(
            f"Original-cohort crosswalk missing {required_link - set(links.columns)}"
        )

    if links[list(required_link)].isna().any().any():
        raise ValueError("Original-cohort crosswalk has incomplete evidence fields")

    if not links.reviewed.str.casefold().eq("true").all():
        raise ValueError("Original-cohort crosswalk includes unreviewed evidence")

    if not links.identity_namespace.eq("GDC_CPTAC3_CASE_SUBMITTER_ID").all():
        raise ValueError(
            "Original-cohort participant identity namespace must be verified "
            "against GDC CPTAC-3 case submitter IDs before an exact join"
        )

    exact_unique(
        ledger, ["case_submitter_id"], "G2 source participant identifiers"
    )

    if not links.case_submitter_id.isin(ledger.case_submitter_id).all():
        raise ValueError("Original-cohort crosswalk contains cases outside G2")

    original_cohort_link_table = links.merge(
        ledger[["case_uuid", "case_submitter_id"]],
        on="case_submitter_id",
        how="left",
        validate="many_to_one",
    )

    # Never silently choose a preferred publication when one patient
    # has been linked to multiple potentially distinct original cohorts.
    per_case = (
        original_cohort_link_table.groupby("case_uuid")
        .agg(
            n_cohorts=("original_cohort", "nunique"),
            n_studies=("original_study_id", "nunique"),
        )
    )
    unambiguous = per_case.index[
        (per_case.n_cohorts == 1) & (per_case.n_studies == 1)
    ]

    one = (
        original_cohort_link_table.loc[
            original_cohort_link_table.case_uuid.isin(unambiguous)
        ]
        .drop_duplicates("case_uuid")
        .set_index("case_uuid")
    )

    for source_col, target_col in [
        ("original_cohort", "original_cohort"),
        ("original_study_id", "original_study_id"),
        ("source_url", "original_cohort_source_url"),
    ]:
        ledger[target_col] = ledger.case_uuid.map(one[source_col])

    ledger.loc[
        ledger.case_uuid.isin(unambiguous),
        "original_cohort_link_status",
    ] = "DOCUMENTED_COHORT_LINK"

    ledger.loc[
        ledger.case_uuid.isin(
            per_case.index.difference(unambiguous)
        ),
        "original_cohort_link_status",
    ] = "AMBIGUOUS_MULTIPLE_COHORT_LINKS"

    evidence_sources["original_cohort_links"] = {
        "path": str(COHORT_LINKS.relative_to(ROOT)),
        "sha256": sha(COHORT_LINKS),
        "n_rows": len(links),
    }

print(
    "Documented original-cohort links:",
    int(ledger.original_cohort_link_status.eq("DOCUMENTED_COHORT_LINK").sum()),
    "/",
    len(ledger),
)


### PDC public case catalog — bounded metadata acquisition

Use the official PDC GraphQL endpoint to retrieve **case identities and external repository references only**. The PDC `allCases` query exposes `case_id`, `case_submitter_id`, project labels and `externalReferences`. <https://pdc.cancer.gov/pdc/publicapi-documentation/>

The first run stores a reusable snapshot of the fetched catalog under `data/interim/metadata/`; later runs reuse it. Set `PDC_REFRESH = True` deliberately to refresh the source. The request is paginated, has a finite page cap, timeout and retry policy, and **fails rather than accepting a partial catalog**. No proteomics, sample payloads, or GDC downloads.

Case links must be **exact and namespace-aware**. A PDC case matching on submitter ID is provisionally linked only when its PDC project identifies CPTAC-3, not CPTAC-2; conflicting or duplicate matches remain unresolved. An exact external GDC case UUID link, when present and clearly labeled as GDC, is stronger. Neither a linked PDC case nor a matching disease label proves a **PDC study membership** or donor-level independence from TCGA.

In [ ]:
# =============================================================================
# Retrieve or reuse the complete PDC public case-identity catalog
# =============================================================================

import time

import requests

PDC_URL = "https://pdc.cancer.gov/graphql"
PDC_SNAPSHOT = META / "113_pdc_public_case_catalog.json"
PDC_REFRESH = False
PDC_PAGE_SIZE = 250
PDC_MAX_PAGES = 100

PDC_FIELDS = """
    case_id
    case_submitter_id
    project_id
    project_submitter_id
    disease_type
    primary_site
    externalReferences {
        external_reference_id
        reference_resource_shortname
        reference_resource_name
        reference_entity_location
    }
"""


def pdc_request(session, query):
    for attempt in range(3):
        try:
            response = session.post(
                PDC_URL,
                json={"query": query},
                headers={"Content-Type": "application/json"},
                timeout=60,
            )
            response.raise_for_status()
            payload = response.json()

            if payload.get("errors"):
                raise ValueError(f"PDC GraphQL errors: {payload['errors']}")

            if not isinstance(payload.get("data"), dict):
                raise ValueError("PDC response missing data object")

            return payload["data"]
        except (requests.RequestException, ValueError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)


if PDC_SNAPSHOT.exists() and not PDC_REFRESH:
    with PDC_SNAPSHOT.open(encoding="utf-8") as handle:
        pdc_snapshot = json.load(handle)
else:
    all_cases = []
    observed_ids = set()

    with requests.Session() as session:
        for page in range(PDC_MAX_PAGES):
            offset = page * PDC_PAGE_SIZE
            query = (
                "{ allCases(offset: " + str(offset)
                + " limit: " + str(PDC_PAGE_SIZE)
                + ") { " + PDC_FIELDS + " } }"
            )
            data = pdc_request(session, query)
            page_cases = data.get("allCases")

            if not isinstance(page_cases, list):
                raise ValueError("PDC allCases returned an unexpected payload")

            for item in page_cases:
                key = item.get("case_id")

                if not key or key in observed_ids:
                    raise ValueError("Missing or repeated PDC case ID")

                observed_ids.add(key)

            all_cases.extend(page_cases)
            print(f"PDC cases collected: {len(all_cases)}")

            if len(page_cases) < PDC_PAGE_SIZE:
                break
        else:
            raise RuntimeError("PDC page cap reached; catalog is incomplete")

    pdc_snapshot = {
        "api": PDC_URL,
        "query": "allCases",
        "page_size": PDC_PAGE_SIZE,
        "cases": all_cases,
    }
    PDC_SNAPSHOT.write_text(
        json.dumps(pdc_snapshot, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

if pdc_snapshot.get("api") != PDC_URL:
    raise ValueError("Unexpected PDC snapshot source")

pdc_cases = pd.DataFrame(pdc_snapshot["cases"])
for name in ["case_id", "case_submitter_id", "project_submitter_id"]:
    if name not in pdc_cases.columns:
        raise ValueError(f"PDC case catalog missing {name}")

if pdc_cases.case_id.isna().any() or pdc_cases.case_id.duplicated().any():
    raise ValueError("Invalid PDC case identity inventory")

print("PDC case-identity snapshot:", len(pdc_cases), "cases; SHA-256:", sha(PDC_SNAPSHOT))


In [ ]:
# =============================================================================
# Diagnose PDC project labels and match GDC candidates across namespaces
# =============================================================================

def normalized_exact(series):
    return series.astype("string").str.strip()


pdc_rows = []

for rec in pdc_snapshot["cases"]:
    refs = rec.get("externalReferences") or []

    if not isinstance(refs, list):
        raise ValueError("Malformed PDC external references")

    gdc_refs = sorted({
        str(ref.get("external_reference_id")).strip().lower()
        for ref in refs
        if (
            "GDC" in str(ref.get("reference_resource_shortname") or "").upper()
            or "GENOMIC DATA COMMONS" in str(
                ref.get("reference_resource_name") or ""
            ).upper()
        )
        and ref.get("external_reference_id")
    })

    pdc_rows.append({
        "pdc_case_id": rec.get("case_id"),
        "case_submitter_id": rec.get("case_submitter_id"),
        "pdc_project": rec.get("project_submitter_id"),
        "pdc_gdc_external_ids": gdc_refs,
    })

pdc_identity = pd.DataFrame(pdc_rows)
exact_unique(pdc_identity, ["pdc_case_id"], "PDC identity catalog")

print("Actual PDC project labels (all projects, not filtered):")
display(
    pdc_identity["pdc_project"]
    .fillna("<MISSING>")
    .value_counts(dropna=False)
    .rename_axis("pdc_project")
    .reset_index(name="n_cases")
)

candidate_ids = set(normalized_exact(ledger.case_submitter_id).dropna())
candidate_uuids = set(normalized_exact(ledger.case_uuid).str.lower().dropna())

matches = []

# Search the *entire* PDC catalog before using labels. PDC project
# nomenclature is not interchangeable with the GDC project ID.
for row in pdc_identity.itertuples(index=False):
    submitter = (
        str(row.case_submitter_id).strip()
        if pd.notna(row.case_submitter_id) else ""
    )

    submitter_hit = submitter in candidate_ids
    uuid_hits = sorted(set(row.pdc_gdc_external_ids) & candidate_uuids)

    if submitter_hit or uuid_hits:
        matches.append({
            "pdc_case_id": row.pdc_case_id,
            "case_submitter_id": submitter,
            "pdc_project": row.pdc_project,
            "gdc_external_case_uuid": "|".join(uuid_hits),
            "match_source": (
                "GDC_EXTERNAL_UUID_AND_SUBMITTER"
                if uuid_hits and submitter_hit
                else "GDC_EXTERNAL_UUID"
                if uuid_hits else "EXACT_SUBMITTER_CROSS_REPOSITORY"
            ),
        })

pdc_matches = pd.DataFrame(matches, columns=[
    "pdc_case_id", "case_submitter_id", "pdc_project",
    "gdc_external_case_uuid", "match_source",
])

print("Exact cross-repository candidate matches:", len(pdc_matches))
if not pdc_matches.empty:
    display(
        pdc_matches.groupby(
            ["pdc_project", "match_source"], dropna=False
        ).size().rename("n_pdc_cases").reset_index()
    )

# Assign a link only when all matching routes uniquely identify the same
# GDC case and exactly one PDC case. Otherwise flag ambiguous evidence.
gdc_by_submitter = ledger.set_index("case_submitter_id")["case_uuid"].to_dict()
candidate_links = []

for rec in pdc_matches.itertuples(index=False):
    linked = set()

    if rec.case_submitter_id in gdc_by_submitter:
        linked.add(gdc_by_submitter[rec.case_submitter_id])

    if rec.gdc_external_case_uuid:
        linked.update(
            uid for uid in rec.gdc_external_case_uuid.split("|")
            if uid in candidate_uuids
        )

    for uid in linked:
        candidate_links.append({
            "case_uuid": uid,
            "pdc_case_id": rec.pdc_case_id,
            "pdc_project": rec.pdc_project,
            "match_source": rec.match_source,
            "conflicting_gdc_ids": len(linked) != 1,
        })

candidate_link_table = pd.DataFrame(candidate_links, columns=[
    "case_uuid", "pdc_case_id", "pdc_project", "match_source",
    "conflicting_gdc_ids",
])

ledger["pdc_case_id"] = pd.NA
ledger["pdc_project"] = pd.NA
ledger["pdc_link_status"] = "UNLINKED_PDC"
ledger["pdc_link_method"] = pd.NA

if not candidate_link_table.empty:
    grouped = candidate_link_table.groupby("case_uuid").agg(
        n_pdc_cases=("pdc_case_id", "nunique"),
        conflicting_ids=("conflicting_gdc_ids", "any"),
    )
    safe_ids = grouped.index[
        (grouped.n_pdc_cases == 1) & ~grouped.conflicting_ids
    ]
    unique = (
        candidate_link_table.loc[
            candidate_link_table.case_uuid.isin(safe_ids)
        ]
        .drop_duplicates("case_uuid")
        .set_index("case_uuid")
    )

    for source, target in [
        ("pdc_case_id", "pdc_case_id"),
        ("pdc_project", "pdc_project"),
        ("match_source", "pdc_link_method"),
    ]:
        ledger[target] = ledger.case_uuid.map(unique[source])

    # Submitter equality is a candidate link, not automatically proof
    # of a shared patient namespace across repositories.
    ledger.loc[
        ledger.case_uuid.isin(safe_ids),
        "pdc_link_status",
    ] = "EXACT_ID_CANDIDATE"

    ledger.loc[
        ledger.case_uuid.isin(grouped.index.difference(safe_ids)),
        "pdc_link_status",
    ] = "AMBIGUOUS_PDC_CASE_LINK"

print("GDC candidate PDC-link status:")
display(ledger.pdc_link_status.value_counts().rename_axis("status").reset_index(name="n_cases"))

evidence_sources["pdc_case_catalog"] = {
    "path": str(PDC_SNAPSHOT.relative_to(ROOT)),
    "sha256": sha(PDC_SNAPSHOT),
    "n_rows": len(pdc_cases),
}


## PDC case → study mapping

The PDC public API exposes `programsProjectsStudies` and `paginatedCaseDemographicsPerStudy` with case UUIDs per PDC study. We use the **study's actual PDC project name**, the exact `pdc_case_id`, and the released latest version to construct case-to-study memberships. <https://pdc.cancer.gov/pdc/publicapi-documentation/>

Only studies under the two *observed* PDC project labels, `CPTAC3 Discovery and Confirmatory` and `CPTAC3-Other`, are requested. Each study has its own `pdc_study_id`, version-specific `study_id` and `study_name`. The same participant can appear in several assays/studies, so **multiple study memberships are not treated as ambiguous patient identity**.

Metadata are snapshotted locally. A bounded request fails on pagination/schema errors; it never substitutes an empty table for a failed query. This establishes PDC study membership, **not original publication equivalence nor TCGA donor independence**.

In [ ]:
# =============================================================================
# Retrieve CPTAC-3 PDC study catalog and participant membership snapshots
# =============================================================================

PDC_STUDY_SNAPSHOT = META / "113_pdc_cptac3_study_memberships.json"
PDC_STUDY_REFRESH = False
PDC_STUDY_PAGE_SIZE = 250
PDC_MAX_STUDIES = 150
PDC_MAX_STUDY_PAGES = 30
PDC_PROJECTS = {
    "CPTAC3 Discovery and Confirmatory",
    "CPTAC3-Other",
}

if PDC_STUDY_SNAPSHOT.exists() and not PDC_STUDY_REFRESH:
    with PDC_STUDY_SNAPSHOT.open(encoding="utf-8") as handle:
        study_snapshot = json.load(handle)
else:
    catalog_query = """
    {
      programsProjectsStudies {
        program_submitter_id
        projects {
          project_submitter_id
          studies {
            pdc_study_id
            study_id
            study_submitter_id
            study_name
          }
        }
      }
    }
    """

    catalog_data = pdc_request(requests.Session(), catalog_query)
    studies = {}

    for program in catalog_data.get("programsProjectsStudies") or []:
        for project in program.get("projects") or []:
            project_name = project.get("project_submitter_id")

            if project_name not in PDC_PROJECTS:
                continue

            for study in project.get("studies") or []:
                study_id = study.get("pdc_study_id")

                if not study_id:
                    raise ValueError("CPTAC-3 PDC study without stable study ID")

                studies.setdefault(study_id, {
                    "pdc_study_id": study_id,
                    "study_id": study.get("study_id"),
                    "study_name": study.get("study_name"),
                    "study_submitter_id": study.get("study_submitter_id"),
                    "pdc_project": project_name,
                })

    if not studies or len(studies) > PDC_MAX_STUDIES:
        raise ValueError(
            f"Unexpected CPTAC-3 study count: {len(studies)}"
        )

    memberships = []
    with requests.Session() as session:
        for study_id, study in sorted(studies.items()):
            collected = 0

            for page in range(PDC_MAX_STUDY_PAGES):
                offset = page * PDC_STUDY_PAGE_SIZE
                query = (
                    '{ paginatedCaseDemographicsPerStudy(pdc_study_id: "'
                    + study_id
                    + '", offset: '
                    + str(offset)
                    + ', limit: '
                    + str(PDC_STUDY_PAGE_SIZE)
                    + ') { total caseDemographicsPerStudy'
                    + ' { case_id case_submitter_id } } }'
                )
                data = pdc_request(session, query)
                response = data.get("paginatedCaseDemographicsPerStudy")

                if not isinstance(response, dict):
                    raise ValueError(
                        f"Missing study participant response for {study_id}"
                    )

                cases = response.get("caseDemographicsPerStudy")
                total = response.get("total")

                if not isinstance(cases, list) or total is None:
                    raise ValueError(
                        f"Missing pagination metadata for {study_id}"
                    )

                for case in cases:
                    if not case.get("case_id"):
                        raise ValueError(
                            f"Missing PDC case ID in {study_id}"
                        )
                    memberships.append({
                        **study,
                        "pdc_case_id": case["case_id"],
                        "pdc_case_submitter_id": case.get("case_submitter_id"),
                    })

                collected += len(cases)

                if collected >= int(total):
                    break

                if not cases:
                    raise ValueError(
                        f"Premature empty participant page for {study_id}"
                    )
            else:
                raise RuntimeError(
                    f"Study {study_id} exceeded pagination cap"
                )

            if collected != int(total):
                raise ValueError(
                    f"Study {study_id} count mismatch: {collected} != {total}"
                )

            print(f"{study_id}: {collected} case memberships")

    study_snapshot = {
        "api": PDC_URL,
        "project_labels": sorted(PDC_PROJECTS),
        "studies": sorted(studies.values(), key=lambda x: x["pdc_study_id"]),
        "memberships": memberships,
    }
    PDC_STUDY_SNAPSHOT.write_text(
        json.dumps(study_snapshot, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

if set(study_snapshot.get("project_labels", [])) != PDC_PROJECTS:
    raise ValueError("Unexpected PDC study project scope in snapshot")

study_memberships = pd.DataFrame(
    study_snapshot["memberships"],
    columns=[
        "pdc_study_id", "study_id", "study_name",
        "study_submitter_id", "pdc_project",
        "pdc_case_id", "pdc_case_submitter_id",
    ],
)

if study_memberships.empty:
    raise ValueError("PDC CPTAC-3 study membership snapshot is empty")

exact_unique(
    study_memberships,
    ["pdc_study_id", "pdc_case_id"],
    "PDC study/participant membership",
)

print(
    "PDC CPTAC-3 study membership catalog:",
    len(study_snapshot["studies"]), "studies;",
    len(study_memberships), "study-case associations",
)


In [ ]:
# =============================================================================
# Join PDC study membership to uniquely linked GDC candidates
# =============================================================================

pdc_case_to_gdc = ledger.loc[
    ledger.pdc_link_status.eq("EXACT_ID_CANDIDATE"),
    ["case_uuid", "pdc_case_id", "pdc_link_method"],
].copy()

if pdc_case_to_gdc.pdc_case_id.duplicated().any():
    raise ValueError(
        "PDC case linked to more than one GDC candidate; manual review required"
    )

study_case_links = study_memberships.merge(
    pdc_case_to_gdc,
    on="pdc_case_id",
    how="inner",
    validate="many_to_one",
)

# Several PDC studies per donor are expected and are all preserved.
linked_studies = study_case_links.groupby("case_uuid").agg(
    n_pdc_studies=("pdc_study_id", "nunique"),
    study_ids=("pdc_study_id", lambda x: "|".join(sorted(set(x)))),
    study_names=("study_name", lambda x: "|".join(sorted(set(x.dropna())))),
)

ledger["n_pdc_studies"] = (
    ledger.case_uuid.map(linked_studies.n_pdc_studies)
    .fillna(0)
    .astype("int64")
)
ledger["pdc_study_ids"] = ledger.case_uuid.map(linked_studies.study_ids)
ledger["pdc_study_membership_status"] = "NO_DOCUMENTED_PDC_STUDY_LINK"

ledger.loc[
    ledger.n_pdc_studies.gt(0),
    "pdc_study_membership_status",
] = "EXACT_PDC_CASE_TO_STUDY"

print(
    "G2 cases with exact PDC case → study membership:",
    int(ledger.n_pdc_studies.gt(0).sum()),
    "/",
    len(ledger),
)

display(
    study_case_links.groupby(
        ["pdc_project", "pdc_study_id", "study_name"], dropna=False
    ).agg(n_g2_cases=("case_uuid", "nunique"))
    .reset_index()
    .sort_values("n_g2_cases", ascending=False)
)

evidence_sources["pdc_study_memberships"] = {
    "path": str(PDC_STUDY_SNAPSHOT.relative_to(ROOT)),
    "sha256": sha(PDC_STUDY_SNAPSHOT),
    "n_rows": len(study_memberships),
}


### G3 exception evidence — optional reviewed overrides

The cohort-level assessment below does not require individual identity investigations. Optional `113_reviewed_donor_evidence.csv` is used **only to record specific positive overlap, verified source exceptions or resolution of such exceptions**, with a source and approval flag. In its absence, the outcome-blind cohort rule is sufficient. Exact identifier non-collisions remain a diagnostic, not a proof of biological identity.

In [ ]:
# =============================================================================
# Apply prospective cohort-level G3 rules and reviewed exceptions
# =============================================================================

# The directly audited GDC origin is the primary provenance source.
if not ledger.project_id.eq("CPTAC-3").all():
    raise ValueError("G3 encountered a non-CPTAC-3 GDC source project")

ledger["g3_class"] = "COHORT_SUPPORTED_EXTERNAL"
ledger["g3_evidence_id"] = "G3_CPTAC3_PROSPECTIVE_COHORT"
ledger["g3_reason"] = (
    "Audited GDC CPTAC-3 acquisition; cohort-level externality "
    "assumption, not individual donor certification"
)
ledger["g3_reviewed"] = True
ledger["g3_source_exception"] = False

# PDC Discovery/Confirmatory reinforces GDC provenance. An unlinked
# PDC case is not automatically excluded if the GDC origin is intact.
# Other PDC subprojects and ambiguous cross-resource identities are
# separately reviewable; they are not silently treated as TCGA overlap.
exception_mask = (
    ledger.pdc_project.notna()
    & ledger.pdc_project.ne("CPTAC3 Discovery and Confirmatory")
) | ledger.pdc_link_status.eq("AMBIGUOUS_PDC_CASE_LINK")

ledger.loc[exception_mask, "g3_class"] = "SOURCE_REVIEW_REQUIRED"
ledger.loc[exception_mask, "g3_evidence_id"] = "G3_PDC_SOURCE_EXCEPTION"
ledger.loc[exception_mask, "g3_reason"] = (
    "PDC project outside Discovery/Confirmatory or ambiguous PDC case "
    "identity; review provenance before external inclusion"
)
ledger.loc[exception_mask, "g3_source_exception"] = True

# A matching identifier alone is a flag, never an automatic donor match.
collision = ledger.uuid_collision_flag | ledger.submitter_collision_flag
ledger.loc[collision, "g3_class"] = "SOURCE_REVIEW_REQUIRED"
ledger.loc[collision, "g3_evidence_id"] = "G3_IDENTIFIER_COLLISION"
ledger.loc[collision, "g3_reason"] = (
    "Exact TCGA identifier collision; assess namespace and provenance"
)
ledger.loc[collision, "g3_source_exception"] = True

if G3_EVIDENCE.exists():
    ev = load(G3_EVIDENCE)
    required_ev = {
        "case_uuid", "g3_class", "evidence_id",
        "evidence_url", "applicability", "reviewed",
    }
    if not required_ev.issubset(ev.columns):
        raise ValueError(f"G3 exception evidence missing {required_ev-set(ev.columns)}")

    exact_unique(ev, ["case_uuid"], "G3 exception evidence")
    if not ev.case_uuid.isin(ledger.case_uuid).all():
        raise ValueError("G3 exception evidence includes outside-G2 cases")

    allowed = {
        "COHORT_SUPPORTED_EXTERNAL",
        "DOCUMENTED_OVERLAP",
        "SOURCE_REVIEW_REQUIRED",
    }
    if not ev.g3_class.isin(allowed).all():
        raise ValueError("Invalid G3 exception classification")
    for field in ["evidence_id", "evidence_url", "applicability"]:
        if ev[field].isna().any() or ev[field].str.strip().eq("").any():
            raise ValueError(f"G3 exception missing {field}")
    if not ev.reviewed.str.casefold().eq("true").all():
        raise ValueError("G3 exception evidence includes unreviewed records")

    ev = ev.set_index("case_uuid")
    for source_col, target_col in [
        ("g3_class", "g3_class"),
        ("evidence_id", "g3_evidence_id"),
        ("applicability", "g3_reason"),
    ]:
        ledger.loc[
            ledger.case_uuid.isin(ev.index), target_col
        ] = ledger.loc[
            ledger.case_uuid.isin(ev.index), "case_uuid"
        ].map(ev[source_col]).to_numpy()

    ledger.loc[ledger.case_uuid.isin(ev.index), "g3_reviewed"] = True
    evidence_sources["g3_exceptions"] = {
        "path": str(G3_EVIDENCE.relative_to(ROOT)),
        "sha256": sha(G3_EVIDENCE),
        "n_rows": len(ev),
    }

assert len(ledger) == 1082 and ledger.case_uuid.is_unique
print("G3 cohort-level result (not individual donor certification):")
display(ledger.g3_class.value_counts().rename_axis("g3_class").reset_index(name="n_cases"))


## G4–G8 — Lineage, assay QC, and analysis-readiness constraints

Source `project_id × primary_site × disease_type` is not interchangeable with harmonized TCGA lineage. Reviewed crosswalks and file-level QC decisions are optional inputs; missing evidence remains `UNMAPPED` or `QC_UNASSESSED`. G6 compatibility and G8 confounder feasibility require separate outcome-blind specification before a cohort can be frozen.

This stage does not estimate program scores, optimize QC cutoffs, or define the Phase 8 replication estimand.

In [ ]:
# =============================================================================
# Adjudicate reviewed source-to-TCGA lineage crosswalk
# =============================================================================

source_keys = ["project_id", "primary_site", "disease_type"]

ledger["tcga_lineage"] = pd.NA
ledger["g4_status"] = "UNMAPPED"
ledger["g4_evidence_id"] = pd.NA

if G4_CROSSWALK.exists():
    cross = load(G4_CROSSWALK)
    req = set(source_keys+["tcga_lineage", "mapping_status", "reviewed", "evidence_id"])
    if not req.issubset(cross.columns):
        raise ValueError(f"G4 crosswalk missing {req-set(cross.columns)}")
    exact_unique(cross, source_keys, "G4 crosswalk")
    if not cross.reviewed.str.casefold().eq("true").all():
        raise ValueError("Unreviewed G4 crosswalk")
    if not cross.mapping_status.isin(["ACCEPTED", "UNMAPPED", "AMBIGUOUS", "CONTRADICTORY"]).all():
        raise ValueError("Invalid G4 status")
    accepted = cross.mapping_status.eq("ACCEPTED")
    if cross.loc[accepted, ["tcga_lineage", "evidence_id"]].isna().any().any():
        raise ValueError("G4 accepted source without lineage/evidence")
    ledger = ledger.merge(cross[source_keys+["tcga_lineage", "mapping_status", "evidence_id"]].rename(columns = {"tcga_lineage": "g4_mapped", "mapping_status": "g4_reviewed_status", "evidence_id": "g4_reviewed_evidence"}), on = source_keys, how = "left", validate = "many_to_one")
    ledger["tcga_lineage"] = ledger["g4_mapped"]
    ledger["g4_status"] = ledger["g4_reviewed_status"].fillna("UNMAPPED")
    ledger["g4_evidence_id"] = ledger["g4_reviewed_evidence"]
    ledger = ledger.drop(columns = ["g4_mapped", "g4_reviewed_status", "g4_reviewed_evidence"])
    evidence_sources["g4"] = {"path": str(G4_CROSSWALK.relative_to(ROOT)), "sha256": sha(G4_CROSSWALK), "n_rows": len(cross)}

In [ ]:
# =============================================================================
# Apply optional reviewed sample-quality evidence
# =============================================================================

ledger["qc_status"] = "QC_UNASSESSED"
ledger["qc_evidence_id"] = pd.NA

if QC_EVIDENCE.exists():
    qc = load(QC_EVIDENCE)
    req = {"file_id", "qc_status", "reviewed", "method_evidence_id"}
    if not req.issubset(qc.columns):
        raise ValueError(f"QC evidence missing {req-set(qc.columns)}")
    exact_unique(qc, ["file_id"], "QC review")
    if not qc.file_id.isin(ledger.file_id).all():
        raise ValueError("QC reviewed file outside G2")
    if not qc.reviewed.str.casefold().eq("true").all():
        raise ValueError("Unreviewed QC status")
    if not qc.qc_status.isin(["QC_PASS", "QC_FAIL", "QC_UNASSESSED"]).all():
        raise ValueError("Invalid QC status")
    if qc.method_evidence_id.isna().any():
        raise ValueError("QC methods evidence ID missing")
    qc = qc.set_index("file_id")
    ledger["qc_status"] = ledger.file_id.map(qc.qc_status).fillna(ledger.qc_status)
    ledger["qc_evidence_id"] = ledger.file_id.map(qc.method_evidence_id)
    evidence_sources["qc"] = {"path": str(QC_EVIDENCE.relative_to(ROOT)), "sha256": sha(QC_EVIDENCE), "n_rows": len(qc)}

In [ ]:
# =============================================================================
# Summarize eligibility blockers without freezing membership
# =============================================================================

ledger["g6_status"] = "INPUT_COMPATIBILITY_UNASSESSED"
ledger["g8_status"] = "CONFOUNDER_FEASIBILITY_UNASSESSED"
ledger["cohort_status"] = "NOT_FROZEN"
ledger["exclusion_or_blocker"] = ledger.apply(lambda r: ";".join([x for x, fail in [
 ("G3_SOURCE_REVIEW", r.g3_class=="SOURCE_REVIEW_REQUIRED"),
 ("G3_OVERLAP", r.g3_class=="DOCUMENTED_OVERLAP"),
 ("G4_UNMAPPED", r.g4_status!="ACCEPTED"),
 ("G5_QC_UNASSESSED", r.qc_status=="QC_UNASSESSED"),
 ("G5_QC_FAIL", r.qc_status=="QC_FAIL"),
 ("G6_UNASSESSED", True), ("G8_UNASSESSED", True)
] if fail]), axis = 1)
assert ledger.case_uuid.is_unique and len(ledger)==1082
display(ledger.groupby(["g3_class", "g4_status", "qc_status"], dropna = False).size().rename("n_cases").reset_index())

## Outputs and interpretation

Persist the case-level eligibility table, documentary linkage ledger, source-lineage mapping, cohort attrition, technical feasibility and SHA-256 manifest. These are **interim evidence outputs** and are not registered as a frozen external cohort.

A successful execution establishes that the G2 candidate inventory is reproducible, **not** that independent-donor or biological validation has passed.

In [ ]:
# =============================================================================
# Define deterministic interim handoff tables
# =============================================================================

outputs = {
    "113_pdc_study_memberships.csv": study_case_links.copy(),
    "113_pdc_study_catalog.csv": pd.DataFrame(study_snapshot["studies"]),
    "113_pdc_case_link_candidates.csv": pdc_matches.copy(),
    "113_pdc_candidate_case_evidence.csv": candidate_link_table.copy(),
    "113_pdc_case_linkage_ledger.csv": ledger[
        ["case_uuid", "case_submitter_id", "pdc_case_id", "pdc_project", "pdc_link_status", "pdc_link_method"]
    ].copy(),
    "113_original_cohort_participant_links.csv": original_cohort_link_table.copy(),
    "113_source_context_to_cohort_candidates.csv": source_context_inventory.copy(),
    "113_donor_identity_and_cohort_links.csv": ledger[
        [
            "case_uuid", "case_submitter_id", "file_id", "sample_uuid",
            "aliquot_uuid", "uuid_collision_flag", "submitter_collision_flag",
            "g3_class", "g3_evidence_id", "g3_reason", "g3_reviewed",
            "g3_source_exception", "pdc_project", "pdc_link_status",
        ]
    ].copy(),
    "113_primary_candidate_case_eligibility.csv": ledger.copy(),
    "113_source_to_tcga_lineage_crosswalk.csv": (
        ledger[source_keys + ["tcga_lineage", "g4_status", "g4_evidence_id"]]
        .drop_duplicates()
        .sort_values(source_keys)
    ),
    "113_cohort_attrition_by_context.csv": (
        ledger.groupby(source_keys, dropna=False)
        .agg(
            g2_simple_cases=("case_uuid", "nunique"),
            g2_primary_files=("file_id", "nunique"),
            n_flagged_ambiguous=("other_ambiguous_rna_flag", "sum"),
            n_documented_independent=(
                "g3_class",
                lambda x: (x == "COHORT_SUPPORTED_EXTERNAL").sum(),
            ),
            n_unresolved_independence=(
                "g3_class",
                lambda x: (x == "SOURCE_REVIEW_REQUIRED").sum(),
            ),
            n_documented_overlap=(
                "g3_class",
                lambda x: (x == "DOCUMENTED_OVERLAP").sum(),
            ),
        )
        .reset_index()
    ),
    "113_assay_qc_and_feature_compatibility.csv": ledger[
        [
            "case_uuid", "file_id", "workflow_type", "provider_md5",
            "qc_status", "qc_evidence_id", "g6_status", "g8_status",
        ]
    ].copy(),
}

In [ ]:
# =============================================================================
# Write evidence tables and source provenance manifest
# =============================================================================

fprints = {}

for name, df in outputs.items():
    path = OUT/name
    df.to_csv(path, index = False)
    fprints[name] = {"sha256": sha(path), "n_rows": len(df)}
inputs = {str(path.relative_to(ROOT)): sha(path) for path in [INDEX, PROVISIONAL, TCGA, REGISTRY]}
manifest = {
    "notebook": 113,
    "input_sha256": inputs,
    "reviewed_evidence": evidence_sources,
    "outputs": fprints,
    "source_counts": {
        "rna_files": 2968,
        "methylation_files": 2610,
        "provisional_primary_cases": 1370,
        "provisional_primary_rna_files": 1731,
        "g2_simple_cases": 1082,
        "multiplicity_cases": 288,
        "multiplicity_primary_files": 649,
        "g2_additional_ambiguous_rna_flagged": 37,
    },
    "cohort_frozen": False,
}

manifest_path = OUT/"113_source_provenance_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent = 2, sort_keys = True)+"\n", encoding = "utf-8")

In [ ]:
# =============================================================================
# Persist and display cohort readiness without analytic authorization
# =============================================================================

status = {
    "notebook": 113,
    "g2_policy": "APPROVED",
    "g3_protocol": "COHORT_LEVEL_PROPORTIONATE",
    "g3_decision_scope": "Cohort-level evidence; no individual donor certification",
    "g3_source_review_cases": int(ledger.g3_class.eq("SOURCE_REVIEW_REQUIRED").sum()),
    "pdc_case_link_counts": ledger.pdc_link_status.value_counts().to_dict(),
    "pdc_study_link_counts": ledger.pdc_study_membership_status.value_counts().to_dict(),
    "g3_class_counts": ledger.g3_class.value_counts().to_dict(),
    "g4_status_counts": ledger.g4_status.value_counts().to_dict(),
    "qc_status_counts": ledger.qc_status.value_counts().to_dict(),
    "g6": "NOT_ADJUDICATED",
    "g8": "NOT_ADJUDICATED",
    "cohort_frozen": False,
    "external_analysis_authorized": False,
    "readiness": "BLOCKED",
    "no_biological_analysis": True,
    "manifest_sha256": sha(manifest_path),
}

status_path = OUT/"113_cohort_readiness.json"
status_path.write_text(json.dumps(status, indent = 2, sort_keys = True)+"\n", encoding = "utf-8")

display(pd.DataFrame([{"artifact": k, **v} for k, v in fprints.items()]))

print("Readiness:", status["readiness"], "; frozen:", status["cohort_frozen"], "; output:", status_path.relative_to(ROOT))

## Decision boundary

- The approved G2 candidate universe is unchanged.
- G3 assesses cohort-level external provenance and documents exceptions; it **does not** claim individual genetic proof of non-overlap.
- GDC CPTAC-3 origin and published prospective acquisition support external-cohort use; retrospective CPTAC–TCGA sources are not accepted.
- Reviewed exceptions and unresolved provenance are retained, never silently rescued.
- G4–G8 and a separate analytic freeze remain pending. Notebook 800 alone owns confirmatory replication.